In [ ]:

# # Hybrid RAG: BM25 + Semantic Search + Reranking
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.document_compressors import CohereRerank
from langchain.retrievers import ContextualCompressionRetriever
from langchain_openai import ChatOpenAI

# Documents
documents = [
    "Python is used for machine learning.",
    "Password reset can be done from account settings.",
    "Machine learning uses algorithms to learn from data.",
    "You can reset your password using the forgot password option."
]

# 1. BM25 Retriever
bm25 = BM25Retriever.from_texts(documents)
bm25.k = 3

# 2. Semantic Retriever
embeddings = OpenAIEmbeddings()

vector_db = FAISS.from_texts(
    documents,
    embeddings
)

semantic = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

# 3. Hybrid Retrieval
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25, semantic],
    weights=[0.5, 0.5]
)

# 4. Reranker
reranker = CohereRerank(
    model="rerank-english-v3.0",
    top_n=2
)

# 5. Combine Hybrid Retrieval + Reranking
reranked_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever
)

# 6. Query
query = "How can I change my password?"

results = reranked_retriever.invoke(query)

# 7. Final relevant documents
for doc in results:
    print(doc.page_content)
# ```

# ### RAG flow

# ```text
# User Query
#     ↓
#  ┌──────────────┐
#  ↓              ↓
# BM25        Semantic Search
#  ↓              ↓
#  └──────┬───────┘
#         ↓
#    Hybrid Retrieval
#         ↓
#      Reranker
#         ↓
#  Top Relevant Chunks
#         ↓
#         LLM
#         ↓
#       Answer
# ```

# **BM25 → keyword matching**
# **Semantic → meaning matching**
# **Hybrid → combines both**
# **Reranker → reorders results by relevance**
# **LLM → generates final answer**


ModuleNotFoundError: No module named 'langchain_community'